In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [2]:
# Đọc dữ liệu
df = pd.read_csv('./dblp-v10.csv')
df.head(3)

,abstract,authors,n_citation,references,title,venue,year,id
0,"In this paper, a robust 3D triangular mesh wat...","['S. Ben Jabra', 'Ezzeddine Zagrouba']",50,"['09cb2d7d-47d1-4a85-bfe5-faa8221e644b', '10aa...",A new approach of 3D watermarking based on ima...,international symposium on computers and commu...,2008,4ab3735c-80f1-472d-b953-fa0557fed28b
1,We studied an autoassociative neural network w...,"['Joaquín J. Torres', 'Jesús M. Cortés', 'Joaq...",50,"['4017c9d2-9845-4ad2-ad5b-ba65523727c5', 'b118...",Attractor neural networks with activity-depend...,Neurocomputing,2007,4ab39729-af77-46f7-a662-16984fb9c1db
2,It is well-known that Sturmian sequences are t...,"['Genevi eve Paquin', 'Laurent Vuillon']",50,"['1c655ee2-067d-4bc4-b8cc-bc779e9a7f10', '2e4e...",A characterization of balanced episturmian seq...,Electronic Journal of Combinatorics,2007,4ab3a4cf-1d96-4ce5-ab6f-b3e19fc260de


In [3]:
df = df.dropna()

In [4]:
main_feature4m_df = df[['abstract', 'authors', 'title', 'venue', 'year']]

In [5]:
# Tạo cột 'paper_details'
main_feature4m_df['paper_details'] = (
    main_feature4m_df['title'] + ' ' + 
    main_feature4m_df['venue'] + ' ' + 
    main_feature4m_df['year'].astype(str) + ' ' + 
    main_feature4m_df['abstract']
)

C:\Users\HP\AppData\Local\Temp\ipykernel_11352\3617426788.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_feature4m_df['paper_details'] = (


In [6]:
# Tính toán TF-IDF
tfidf_vec = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2, ngram_range=(1, 1))
tfidf_matrix = tfidf_vec.fit_transform(main_feature4m_df['paper_details'])

In [7]:
# Tính toán K-NN
knn = NearestNeighbors(n_neighbors=5, metric='cosine')  # Sử dụng cosine distance
knn.fit(tfidf_matrix)

NearestNeighbors(metric='cosine')

In [8]:
# Hàm gợi ý tài liệu
def content_based_knn_recommendations(paper_index, data, knn_model):
    distances, indices = knn_model.kneighbors(tfidf_matrix[paper_index], n_neighbors=10)  
    return data.iloc[indices.flatten()[1:]], distances.flatten()[1:]

# Ví dụ gọi hàm với chỉ số bài báo
input_paper_index = int(input("index of the paper : "))
recommendations_result, distances = content_based_knn_recommendations(input_paper_index, main_feature4m_df, knn)

# Hiển thị kết quả
print("Recommendations:")
print(recommendations_result[['title']])

Recommendations:
                                                    title
488154  Competition Between Synaptic Depression and Fa...
308304              Neural networks with dynamic synapses
104939  Depression-Biased Reverse Plasticity Rule Is R...
475412  Neuronal regulation: a mechanism for synaptic ...
17361   Observed Stent's anti-Hebbian postulate on dyn...
432670  Processing of Time Series by Neural Circuits w...
822456  The Enhanced Rise and Delayed Fall of Memory i...
787197  Synaptic depression in deep neural networks fo...
70276   A VLSI recurrent network of integrate-and-fire...
